# Using the harness from a notebook

The `bench` package has no dependency on the web app and none on the CLI, so
everything the benchmark does is available here as plain function calls.
This notebook is the short tour.

- `docs/architecture.md` - what the pieces are and what runs where
- `notebooks/practice/StereoAlgo1.ipynb` - the original hand-written matcher
  this harness grew out of

In [ ]:
import sys
from pathlib import Path

# Make `import bench` work whether this runs from the repo root or notebooks/.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bench").is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from bench import paths

print(f"repo root: {paths.REPO_ROOT}")
print(f"middlebury downloaded: {paths.MIDDLEBURY_DIR.is_dir()}")
print(f"onnx weights present:  {any(paths.MODELS_DIR.glob('*.onnx'))}")

## What is registered

Algorithms self-register, so the live set is whatever `bench.algorithms`
imported. `availability()` never raises - it reports a missing optional
dependency as a reason string instead.

In [ ]:
from bench.algorithms import all_specs

for spec in all_specs():
    ok, why = spec.availability()
    print(f"{'  ' if ok else '! '}{spec.family:8s} {spec.name:30s} {why}")

## One pair, one matcher

A synthetic pair needs no download and its ground truth is **exact** by
construction - the right view is the left warped by a known disparity field,
rather than a measurement. Swap in `load_middlebury_scene("Teddy")` once
`scripts/get_dataset.py` has run.

In [ ]:
from bench.algorithms import select
from bench.datasets import make_synthetic_pair

pair = make_synthetic_pair(seed=0, size=(256, 320), ndisp=48)
spec = select(["scratch_census_wta"])[0]
result = spec.build().compute(pair)

disp, valid = result.normalized()
print(pair)
print(f"answered {100 * valid.mean():.1f}% of pixels")

## Scoring it

`evaluate` reports every error metric **twice**: over the whole evaluation
region with missing estimates counted as errors (`bad_all`, the ranking
metric), and over only the pixels the method answered (`bad_valid`). Read the
second next to `density`, never alone - scoring only where a method chose to
answer rewards methods that answer rarely.

In [ ]:
from bench.metrics import evaluate

# `valid` is a separate argument, not folded into `disp`: a method's own
# validity mask and the scene's evaluation region are different things.
acc = evaluate(disp, valid, pair.gt, eval_mask=pair.occ_mask)
print(f"bad 2.0 (all pixels)     {acc.bad_all['2']:.2f}%")
print(f"bad 2.0 (answered only)  {acc.bad_valid['2']:.2f}%")
print(f"density                  {100 * acc.density:.1f}%")
print(f"end-point error          {acc.epe_valid:.3f} px")

In [ ]:
import matplotlib.pyplot as plt

from bench.metrics import colorize

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
vmax = float(pair.ndisp)
panels = [
    ("left", pair.left),
    ("ground truth", colorize(pair.gt, 0, vmax)),
    ("estimated", colorize(disp, 0, vmax)),
]
for ax, (title, img) in zip(axes, panels):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## A whole grid

`run_benchmark` is what `python -m bench run` calls. Point `out_dir`
somewhere temporary from a notebook - the default writes a timestamped folder
into `results/`, and an exploratory run is not worth keeping.

Cells run **sequentially** on purpose: the harness measures time and CPU, and
two matchers competing for the same cores would make both meaningless.

In [ ]:
import tempfile

from bench.report import leaderboard, to_dataframe
from bench.runner import RunConfig, run_benchmark

specs = select(["opencv_sgbm_3way", "scratch_census_wta", "scratch_sad_wta"])
config = RunConfig(
    repeats=1,
    warmup=1,
    save_figures=False,
    out_dir=Path(tempfile.mkdtemp(prefix="bench-notebook-")),
)
cells, out_dir = run_benchmark([pair], specs, config, verbose=False)

leaderboard(to_dataframe(cells))

## Reading a finished run back

`results.csv` is the interface - there is no database, and `pandas.read_csv`
is the query language. One row per (scene, algorithm), every metric a column.

The run below is the committed one quoted in the root README; see
`results/keep/README.md`.

In [ ]:
import pandas as pd

kept = paths.KEPT_RESULTS_DIR / "20261105-035941" / "results.csv"
df = pd.read_csv(kept)

print(f"{len(df)} cells, {df.algorithm.nunique()} algorithms, "
      f"{df.scene.nunique()} scenes")
(df[df.status == "ok"]
 .groupby("algorithm")[["bad2_all", "wall_s", "cpu_s"]]
 .mean()
 .sort_values("bad2_all")
 .head(10))